# XGBoost Retrain â€” Save as joblib

Retrains XGBoost **only** (RF not touched). Uses the best hyperparameters already found in `02_model_training_final.ipynb` so no GridSearchCV is needed.

Output goes to `model/xgb_joblib/` in Drive â€” separate from other model files.

**Why joblib instead of `.ubj`?** Joblib serialises the entire Python `XGBRegressor` object (base_score, trees, all internal state) exactly as it is in memory. RF has always worked this way. Loading it back gives an identical object â€” no XGBoost version compatibility issues, no base_score loss.

In [ ]:
!pip install xgboost scikit-learn joblib --quiet
print('Packages ready.')

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json as json_lib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/CST3990 - Final Year Project'
DATA_DIR   = f'{DRIVE_BASE}/data'
MODEL_DIR  = f'{DRIVE_BASE}/model'           # existing files stay here
OUT_DIR    = f'{DRIVE_BASE}/model/xgb_joblib' # clean output folder
os.makedirs(OUT_DIR, exist_ok=True)

print(f'Output folder: {OUT_DIR}')

## Load data

In [ ]:
df_harvest = pd.read_csv(f'{DATA_DIR}/season_end_data.csv')
df_sat     = pd.read_csv(f'{MODEL_DIR}/satellite_features.csv')

print(f'Harvest:   {df_harvest.shape}')
print(f'Satellite: {df_sat.shape}')

df = pd.merge(
    df_harvest[['season', 'region', 'tch', 'surface_harvested', 'cane_production']],
    df_sat,
    on=['season', 'region'],
    how='inner'
)
print(f'Merged:    {df.shape}')

## Feature engineering (identical to main notebook)

In [ ]:
MONTH_SUFFIX     = ['june', 'july', 'aug', 'sep', 'oct', 'nov', 'dec']
ndvi_monthly     = [f'ndvi_{m}'     for m in MONTH_SUFFIX]
rainfall_monthly = [f'rainfall_{m}' for m in MONTH_SUFFIX]
temp_monthly     = [f'temp_{m}'     for m in MONTH_SUFFIX]

# Impute missing monthly NDVI with row ndvi_mean
for col in ndvi_monthly:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df['ndvi_mean'])

# Impute missing rainfall / temperature with column mean
for col in rainfall_monthly + temp_monthly:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mean())

# Previous-season harvested area (survivor effect)
df = df.sort_values(['region', 'season']).reset_index(drop=True)
df['surface_prev'] = df.groupby('region')['surface_harvested'].shift(1)
mask_2008 = df['season'] == 2008
df.loc[mask_2008, 'surface_prev'] = df.loc[mask_2008, 'surface_harvested']

# One-hot encode region (CENTRE = baseline) and satellite (landsat7 = baseline)
df['reg_name'] = df['region']
df_encoded = pd.get_dummies(df, columns=['region'],    drop_first=True, dtype=int)
df_encoded = pd.get_dummies(df_encoded, columns=['satellite'], drop_first=True, dtype=int)
df_encoded['reg_name'] = df['reg_name']

region_dummies = sorted([c for c in df_encoded.columns if c.startswith('region_')])
sat_dummies    = sorted([c for c in df_encoded.columns if c.startswith('satellite_')])

NDVI_FEATURES = ndvi_monthly + ['ndvi_mean', 'ndvi_max', 'ndvi_cumulative']

ALL_FEATURES = (
    NDVI_FEATURES
    + region_dummies
    + sat_dummies
    + rainfall_monthly
    + temp_monthly
    + ['surface_prev']
)
TARGET = 'tch'

assert len(ALL_FEATURES) == 31, f'Expected 31 features, got {len(ALL_FEATURES)}'
print(f'Features: {len(ALL_FEATURES)}  (check: OK)')
print(f'Region dummies:    {region_dummies}')
print(f'Satellite dummies: {sat_dummies}')

## Train / holdout split

In [ ]:
train_df = df_encoded[df_encoded['season'] <= 2024].copy()
test_df  = df_encoded[df_encoded['season'] == 2025].copy()

X_train       = train_df[ALL_FEATURES].values
y_train       = train_df[TARGET].values
X_test        = test_df[ALL_FEATURES].values
y_test        = test_df[TARGET].values
train_seasons = train_df['season'].values

print(f'Training set: {X_train.shape}  (2008-2024)')
print(f'Test set:     {X_test.shape}   (2025 holdout)')
print(f'y_train mean: {y_train.mean():.6f}')

## LOSO evaluation helper

In [ ]:
def loso_evaluate(model, X, y, seasons_array):
    unique_seasons = sorted(np.unique(seasons_array))
    all_preds = np.zeros(len(y))
    for held_out in unique_seasons:
        train_idx = seasons_array != held_out
        test_idx  = seasons_array == held_out
        model.fit(X[train_idx], y[train_idx])
        all_preds[test_idx] = model.predict(X[test_idx])
    rmse = np.sqrt(mean_squared_error(y, all_preds))
    mae  = mean_absolute_error(y, all_preds)
    r2   = r2_score(y, all_preds)
    return {'predictions': all_preds, 'actual': y, 'rmse': rmse, 'mae': mae, 'r2': r2}

## Train XGBoost

Best hyperparameters confirmed from the original GridSearchCV run â€” no need to search again.

In [ ]:
BEST_XGB_PARAMS = {
    'colsample_bytree': 0.7,
    'learning_rate':    0.1,
    'max_depth':        3,
    'min_child_weight': 1,
    'n_estimators':     100,
    'subsample':        0.8,
}

# base_score must be set explicitly from training data.
# When left as None (auto), XGBoost computes it correctly in memory
# but loses it during any serialisation (joblib pickle calls save_raw
# internally, which has the same bug as save_model). Setting it
# explicitly means the value is stored as a normal float attribute.
BASE_SCORE = float(y_train.mean())
print(f"base_score = {BASE_SCORE:.6f}  (mean TCH over 2008-2024 training set)")

# LOSO cross-validation to confirm metrics match the original run
xgb_cv = XGBRegressor(**BEST_XGB_PARAMS, base_score=BASE_SCORE,
                       random_state=RANDOM_SEED, tree_method="hist", verbosity=0)
xgb_loso = loso_evaluate(xgb_cv, X_train, y_train, train_seasons)

print("XGBoost LOSO CV Results:")
print(f"  RMSE: {xgb_loso["rmse"]:.4f}  (expected 6.3738)")
print(f"  MAE:  {xgb_loso["mae"]:.4f}  (expected 4.7587)")
print(f"  R²:   {xgb_loso["r2"]:.4f}  (expected 0.6522)")

In [ ]:
# Train final model on full 2008-2024
final_xgb = XGBRegressor(**BEST_XGB_PARAMS, base_score=BASE_SCORE,
                          random_state=RANDOM_SEED, tree_method="hist", verbosity=0)
final_xgb.fit(X_train, y_train)
print(f"Final XGBoost trained. base_score={final_xgb.base_score:.6f}")

## 2025 holdout evaluation

In [ ]:
xgb_pred_2025 = final_xgb.predict(X_test)

results_2025 = pd.DataFrame({
    'region':        test_df['reg_name'].values,
    'actual_tch':    y_test,
    'xgb_predicted': xgb_pred_2025,
    'error':         xgb_pred_2025 - y_test,
})
print('2025 Holdout Results:')
print(results_2025.to_string(index=False, float_format='%.2f'))

xgb_rmse_2025 = float(np.sqrt(mean_squared_error(y_test, xgb_pred_2025)))
xgb_r2_2025   = float(r2_score(y_test, xgb_pred_2025))
print(f'\n2025 Holdout  RMSE={xgb_rmse_2025:.4f}  R\u00b2={xgb_r2_2025:.4f}')

## Save as joblib

Joblib serialises the entire `XGBRegressor` Python object â€” identical to how the Random Forest model is saved. No `.ubj` format, no base_score loss.

In [ ]:
XGB_PATH = f"{OUT_DIR}/xgb_model.joblib"
joblib.dump(final_xgb, XGB_PATH)
print(f"Saved: {XGB_PATH}")

# Verify round-trip: load and predict — values must match AND be realistic
verify       = joblib.load(XGB_PATH)
verify_preds = verify.predict(X_test)

print(f"Loaded model base_score: {verify.base_score:.6f}")
print("
Verification (loaded model predictions):")
for region, pred, actual in zip(test_df["reg_name"].values, verify_preds, y_test):
    print(f"  {region:<8} predicted={pred:.2f}  actual={actual:.2f}")

assert np.allclose(xgb_pred_2025, verify_preds, atol=0.01), "Round-trip mismatch!"
assert verify_preds.min() > 20, f"Predictions look wrong (min={verify_preds.min():.2f}) — base_score not saved?"
print("
Round-trip check: OK — predictions match and are in realistic range.")

In [ ]:
metadata = {
    'feature_columns': ALL_FEATURES,
    'target': TARGET,
    'training_seasons': list(range(2008, 2025)),
    'test_season': 2025,
    'best_params': BEST_XGB_PARAMS,
    'loso_rmse': float(xgb_loso['rmse']),
    'loso_mae':  float(xgb_loso['mae']),
    'loso_r2':   float(xgb_loso['r2']),
    'test_rmse': xgb_rmse_2025,
    'test_r2':   xgb_r2_2025,
    'base_score':  BASE_SCORE,
    'save_format': 'joblib',
    'note': 'Joblib preserves full XGBRegressor object. Same approach as rf_model.joblib.'
}
with open(f'{OUT_DIR}/metadata.json', 'w') as fh:
    json_lib.dump(metadata, fh, indent=2)

print(f'Files in {OUT_DIR}:')
for f in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(f'{OUT_DIR}/{f}') / 1024
    print(f'  {f:<35} {size:>8.1f} KB')

print('\nDone. Download xgb_model.joblib and place it in rekolte-api/models/')
print('Then update MongoDB model_config: filepath -> "xgb_model.joblib"')